# Train on a track zoo

The workflow in four steps: **declare** the world as a manifest, **see**
it, **split** it into train/test, and **train** on the train half — with
per-track evaluation on the held-out tracks running automatically at the
end of training. Results land under `runs/`; continue in
`run_analysis.ipynb` afterwards.


In [ ]:
import os

# must be set before CUDA init (stacked camera obs fragment 8 GB GPUs)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if os.getcwd().endswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))

from deepracer_genesis.tools.zoo import OfficialSample, Zoo, compile_zoo


## 1. Declare the world

A `Zoo` manifest is config-as-code. `OfficialSample(12, seed=7)` samples
12 ORIGINAL DeepRacer tracks (fetched + cached on first use), adds gentle
waypoint noise, and randomizes each clone's look (palette / field /
wall). Fully offline alternative:
`from deepracer_genesis.tools.zoo import demo_zoo`.


In [ ]:
population = Zoo("nb_population", (
    OfficialSample(12, seed=7, jitter=0.4, looks=True),
))

names = compile_zoo(population)   # bakes once, cached by deterministic names
names


## 2. See it before you train on it

One overview render of every tile (in a GUI session use
`view(population)` instead and orbit with the mouse; `watch(population)`
adds driving cars and saves car-view contact sheets).


In [ ]:
from IPython.display import Image as NBImage
from deepracer_genesis.tools.zoo import view_zoo

view_zoo(names, screenshot="logs/zoo/nb_overview.png")
NBImage("logs/zoo/nb_overview.png", width=900)


## 3. Train/test split

`TrackDataset` splits deterministically (hash-ranked by seed — stable
when other tracks are added later). Test names go to the END-OF-TRAINING
per-track evaluation, never to training.


In [ ]:
from deepracer_genesis.datasets.splits import TrackDataset

split = TrackDataset(names=names, holdout=(), test_fraction=0.25, seed=0)
print(f"train ({len(split.train)}): {split.train}")
print(f"test  ({len(split.test)}): {split.test}")


## 4. Put it to train

The experiment is the same config-as-code as everything else.
`Evaluation(real_tracks=...)` runs the per-track holdout eval after
training finishes and saves charts with the run. Adjust
`total_env_steps` / `num_envs` to your GPU; this cell is the long one.


In [ ]:
from deepracer_genesis.experiment import (
    PPO, AsymmetricCameraPolicy, CameraEnvironment,
    DomainRandomizationActions, DomainRandomizationCamera,
    DomainRandomizationPhysics, DomainRandomizationTrackAppearance,
    Evaluation, Experiment, run,
)


class ZooNotebook(Experiment):
    """Camera policy on the notebook zoo with full obs-side DR."""

    total_env_steps = 10_000_000
    eval_every_steps = 2_000_000
    ablation_group = "notebooks"
    variant = "zoo_notebook"

    def pipeline(self):
        return (
            CameraEnvironment(render="madrona", resolution=(160, 120),
                              num_envs=64, tracks=split.train,
                              random_direction=True)
            >> DomainRandomizationTrackAppearance(strength=0.6)
            >> DomainRandomizationCamera(brightness=(0.7, 1.3), hue=0.05,
                                         blur=0.3, camera_jitter=True)
            >> DomainRandomizationPhysics()
            >> AsymmetricCameraPolicy(actor_keys=("camera",),
                                      critic_keys=("camera", "state"))
            >> DomainRandomizationActions(steer_noise=0.02, speed_noise=0.05,
                                          delay_steps=1)
            >> PPO(minibatches=8)
            >> Evaluation(real_tracks=split.test, charts=True)
        )


record = run(ZooNotebook)
record.metrics


Training writes TensorBoard events, `model.pt`, `eval_record.json`, and
`charts/` into the run directory printed above — continue in
**`run_analysis.ipynb`**.
